In [ ]:
import geopandas as gpd                               # práce s geospatial daty
import matplotlib.pyplot as plt                       # vizualizace
import osmnx as ox                                    # práce s  Openstreetmap daty
import osm2geojson                                    # práce s Openstreetmap daty
import io                                             # práce s daty, bez nutnosti ukládat na disk
import pandas as pd                                   # data manipulation a analýza
import folium                                         # interaktivní mapy
import contextily as ctx                              # basemaps
import requests                                       # Requesty na weby
from bs4 import BeautifulSoup                         # Scraping dat
from shapely.geometry import Polygon, shape, Point    # práce s polygony

In [ ]:
pd.set_option('display.max_columns', None)            # Zajišťuje že vidím všechny sloupce

In [ ]:
seznam_kraju = [
    "Hlavní město Praha", "Středočeský kraj", "Jihočeský kraj", "Plzeňský kraj", 
    "Karlovarský kraj", "Ústecký kraj", "Liberecký kraj", "Královéhradecký kraj", 
    "Pardubický kraj", "Kraj Vysočina", "Jihomoravský kraj", "Olomoucký kraj", 
    "Zlínský kraj", "Moravskoslezský kraj"]
kraje_geometrie = ox.geocode_to_gdf(seznam_kraju)
print(kraje_geometrie[['display_name', 'geometry']].head())

In [ ]:
Obce_data = requests.get("https://vdp.cuzk.gov.cz/vdp/ruian/obce?sort=NAZEV&ohradaId=&nespravny=&kodVc=&kodOk=&kodOp=&kodPu=&nazevOb=&statusKod=&search=&mediaType=csv")
Obce_data = Obce_data.content.decode('utf-8', errors='replace')
Obce_data = pd.read_csv(io.StringIO(Obce_data), sep=';')

In [ ]:
Exekuce_data = requests.get("http://statistiky.ekcr.info/otevrena-data/data/obce.csv")
Exekuce_data = Exekuce_data.content.decode('utf-8', errors='replace')
Exekuce_data = pd.read_csv(io.StringIO(Exekuce_data), sep=',').rename(columns={"pocet_osob" : "Osob v exekuci" })

In [ ]:
Obyvatele_data = requests.get("https://csu.gov.cz/docs/107508/ebed5ef3-dca1-baf2-c0d3-824b0893086f/1300722503.xlsx?version=1.0")
Obyvatele_data = pd.read_excel(io.BytesIO(Obyvatele_data.content), skiprows=5)
Obyvatele_data = Obyvatele_data.iloc[:,1:]
Obyvatele_data.columns = ["Kód obce", "Obec", "Obyvatel_celkem", "Mužů", "Žen", "Průměrný věk","Průměrný věk muži","Průměrný věk ženy"]
Obyvatele_data = Obyvatele_data.dropna(subset=["Kód obce"])

In [ ]:
vysledny_df = (
    Obce_data
    .merge(Exekuce_data, left_on='Kód', right_on='kod_obce_zuj', how='inner').drop(columns=["kod_obce_zuj"])
    .merge(Obyvatele_data, left_on='Kód', right_on='Kód obce', how='inner')
    .drop(columns=["Kód obce", "okres","nazev_obce", "Název Kraje (VÚSC)", "Obec", "obvod_orp"])
    .rename(columns={"Kód" : "Kód obce"}))

In [ ]:
# Výpočet % exekucí v jednotlivých krajích
kraje_summary = vysledny_df.groupby('kraj')[['Osob v exekuci', 'Obyvatel_celkem']].sum().reset_index()
kraje_summary['procento_exekuci'] = (kraje_summary['Osob v exekuci'] / kraje_summary['Obyvatel_celkem']) * 100
df_kraj_exekuce = kraje_summary[['kraj', 'procento_exekuci']]

In [ ]:
df_kraj_exekuce

In [ ]:
#vysledny_df.shape[1]

In [ ]:
#vysledny_df[vysledny_df['Název POU'].str.contains('Blansko', na=False)].head()

In [ ]:
#Obce_data[Obce_data['Název Obce'] == 'Lovosice']

In [ ]:
#vysledny_df[vysledny_df['Název Okresu'].isna()]

In [ ]:
#print(vysledny_df.isna().sum())            # NA check